# z005 - Optuna con la receta del profesor

**Objetivo:** encontrar los mejores `num_leaves`, `min_sum_hessian_in_leaf` y cantidad de árboles.

1. **Optuna, 40 trials** con 1 semilla (501419). Train 202104, valid 202106, variables con lags 1 y 2.
   Cada trial se guarda en SQLite (`C:/Users/keila/optuna/z005.db`): si se corta, se retoma.
2. **Revalidación:** los 10 mejores trials se vuelven a entrenar con las 5 semillas → se ordenan por el promedio
   (con 1 sola semilla el ranking puede ser suerte).

Ojo: el early stopping y la ganancia miran 202106 → los números son algo optimistas (igual para todos los trials).

In [1]:
# --- 1. Librerías y parámetros ---
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna

GANANCIA_ACIERTO = 1_072_500
COSTO_ESTIMULO   = 27_500
MESES_TRAIN = [202104]
MES_VALID   = 202106
SEMILLA     = 501419
SEMILLAS    = [501419, 623353, 786349, 834569, 946331]

N_TRIALS    = 40
TOP_REVALIDAR = 10
MAX_ARBOLES = 3000

# Base de datos fuera de OneDrive (la sincronización puede trabar el archivo)
STORAGE = "sqlite:///C:/Users/keila/optuna/z005.db"
ESTUDIO = "z005_profesor_s501419"   # nombre nuevo: si cambia la semilla, cambiar el nombre

# Fijos del profesor
params_fijos = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.005,
    "min_data_in_leaf": 0,
    "feature_fraction": 0.5,
    "feature_fraction_bynode": 0.2,
    "max_bin": 31,
    "verbose": -1,
}

In [2]:
# --- 2. Cargar datos, target y lags (igual que z003) ---
df = pd.read_parquet("../competencia_01.parquet")
df["clase_binaria"] = (df["clase_ternaria"] == "BAJA+2").astype(int)

no_usar = ["foto_mes", "clase_ternaria", "clase_binaria"]
originales = [c for c in df.columns if c not in no_usar]
cols = [c for c in originales if c != "numero_de_cliente"]

df = df.sort_values(["numero_de_cliente", "foto_mes"]).reset_index(drop=True)
grupo = df.groupby("numero_de_cliente")[cols]
nuevas = []
for k in [1, 2]:
    lag = grupo.shift(k).astype("float32")
    delta = (df[cols] - lag).astype("float32")
    lag.columns   = [f"{c}_lag{k}" for c in cols]
    delta.columns = [f"{c}_delta{k}" for c in cols]
    nuevas += [lag, delta]
df = pd.concat([df] + nuevas, axis=1)
del nuevas, lag, delta, grupo

variables = [c for c in df.columns if c not in no_usar]
train = df[df["foto_mes"].isin(MESES_TRAIN)]
valid = df[df["foto_mes"] == MES_VALID]
y_valid = valid["clase_binaria"].values
ganancia_cliente = np.where(y_valid == 1, GANANCIA_ACIERTO, -COSTO_ESTIMULO)

# Datasets de LightGBM armados una sola vez (se reusan en todos los trials)
datos_train = lgb.Dataset(train[variables], label=train["clase_binaria"], free_raw_data=False)
datos_valid = lgb.Dataset(valid[variables], label=valid["clase_binaria"], reference=datos_train, free_raw_data=False)
print("Variables:", len(variables))

Variables: 761


In [3]:
# --- 3. Funciones: ganancia y objetivo de Optuna ---
def ganancias(prob):
    # Ordenamos de mayor a menor probabilidad y acumulamos la ganancia
    acum = np.cumsum(ganancia_cliente[np.argsort(prob)[::-1]])
    n = int(np.argmax(acum)) + 1
    return {"ganancia_max_M": acum[n - 1] / 1e6, "mejor_n": n,
            "top11500_M": acum[11_499] / 1e6, "top13000_M": acum[12_999] / 1e6}

def objetivo(trial):
    # Los 3 hiperparámetros que el profesor manda optimizar
    p = {**params_fijos,
         "num_leaves": trial.suggest_int("num_leaves", 8, 256, log=True),
         "min_sum_hessian_in_leaf": trial.suggest_float("min_sum_hessian_in_leaf", 1e-3, 10, log=True),
         "seed": SEMILLA}
    # La cantidad de árboles la elige el early stopping (corta si el AUC no mejora en 200)
    modelo = lgb.train(p, datos_train, num_boost_round=MAX_ARBOLES, valid_sets=[datos_valid],
                       callbacks=[lgb.early_stopping(200, verbose=False)])
    best_iter = modelo.best_iteration or modelo.num_trees()
    r = ganancias(modelo.predict(valid[variables], num_iteration=best_iter))

    # Guardamos datos extra en el trial (z601: usar el best_iter real en el modelo final)
    trial.set_user_attr("best_iter", best_iter)
    trial.set_user_attr("arboles_entrenados", modelo.num_trees())
    trial.set_user_attr("mejor_n", r["mejor_n"])
    trial.set_user_attr("top11500_M", round(r["top11500_M"], 1))
    return r["ganancia_max_M"]

In [4]:
# --- 4. Correr Optuna (retoma si ya hay trials guardados) ---
optuna.logging.set_verbosity(optuna.logging.WARNING)
estudio = optuna.create_study(
    study_name=ESTUDIO, storage=STORAGE, load_if_exists=True,
    direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEMILLA),
)
hechos = len([t for t in estudio.trials if t.state == optuna.trial.TrialState.COMPLETE])
print("Trials ya hechos:", hechos)
estudio.optimize(objetivo, n_trials=max(0, N_TRIALS - hechos))

trials = estudio.trials_dataframe(attrs=("number", "value", "params", "user_attrs", "state"))
trials = trials[trials["state"] == "COMPLETE"].sort_values("value", ascending=False)
trials.to_csv("z005_optuna_trials.csv", sep=";", decimal=",", index=False, encoding="utf-8-sig")
print("Mejor ganancia:", round(estudio.best_value, 1), "M | params:", estudio.best_params,
      "| best_iter:", estudio.best_trial.user_attrs["best_iter"])
trials.head(10)

Trials ya hechos: 0


Mejor ganancia: 401.8 M | params: {'num_leaves': 37, 'min_sum_hessian_in_leaf': 0.0031923994080982777} | best_iter: 868


,number,value,params_min_sum_hessian_in_leaf,params_num_leaves,user_attrs_arboles_entrenados,user_attrs_best_iter,user_attrs_mejor_n,user_attrs_top11500_M,state
6,6,401.7750,0.003192,37,868,868,13510,370.2,COMPLETE
34,34,400.5100,9.755574,137,538,538,12076,391.0,COMPLETE
27,27,399.4375,5.632060,241,578,578,12315,395.4,COMPLETE
21,21,398.5850,1.240045,231,648,648,14026,380.0,COMPLETE
36,36,397.3750,7.745374,61,727,727,15390,391.0,COMPLETE
37,37,395.5875,0.003039,240,888,888,12655,385.6,COMPLETE
24,24,395.3400,2.344906,250,572,572,12904,390.0,COMPLETE
10,10,393.9925,0.029433,10,894,894,15753,324.0,COMPLETE
29,29,393.2775,5.968488,193,567,567,11259,390.0,COMPLETE
9,9,393.2225,1.250592,65,684,684,15141,381.2,COMPLETE


In [5]:
# --- 5. Revalidar los 10 mejores con las 5 semillas ---
# Mismos hiperparámetros y cantidad de árboles (best_iter) del trial, cambiando solo la semilla
filas = []
for _, t in trials.head(TOP_REVALIDAR).iterrows():
    p = {**params_fijos,
         "num_leaves": int(t["params_num_leaves"]),
         "min_sum_hessian_in_leaf": float(t["params_min_sum_hessian_in_leaf"])}
    for semilla in SEMILLAS:
        modelo = lgb.train({**p, "seed": semilla}, datos_train, num_boost_round=int(t["user_attrs_best_iter"]))
        filas.append({"trial": int(t["number"]), "semilla": semilla, **ganancias(modelo.predict(valid[variables]))})
    print("Trial", int(t["number"]), "revalidado")

reval = pd.DataFrame(filas)
ranking = (reval.groupby("trial")[["ganancia_max_M", "top11500_M", "top13000_M", "mejor_n"]]
           .agg(["mean", "std"]).round(1))
ranking.columns = ["_".join(c) for c in ranking.columns]
ranking = ranking.join(trials.set_index("number")[["value", "params_num_leaves",
                       "params_min_sum_hessian_in_leaf", "user_attrs_best_iter"]].rename(columns={"value": "optuna_1semilla_M"}))
ranking = ranking.sort_values("ganancia_max_M_mean", ascending=False)
ranking.to_csv("z005_top10_5semillas.csv", sep=";", decimal=",", encoding="utf-8-sig")
ranking

Trial 6 revalidado


Trial 34 revalidado


Trial 27 revalidado


Trial 21 revalidado


Trial 36 revalidado


...

Trial 24 revalidado


Trial 10 revalidado


Trial 29 revalidado


Trial 9 revalidado


,ganancia_max_M_mean,ganancia_max_M_std,top11500_M_mean,top11500_M_std,top13000_M_mean,top13000_M_std,mejor_n_mean,mejor_n_std,optuna_1semilla_M,params_num_leaves,params_min_sum_hessian_in_leaf,user_attrs_best_iter
trial,,,,,,,,,,,,
34,399.6,2.3,392.2,3.2,394.5,3.7,13212.0,1072.1,400.5100,137,9.755574,538
36,395.5,4.4,391.7,4.5,383.7,5.4,12291.2,1744.0,397.3750,61,7.745374,727
27,394.6,3.8,390.2,4.2,386.5,6.1,12740.2,1703.9,399.4375,241,5.632060,578
29,394.5,2.0,388.0,2.7,390.1,3.1,12735.8,1313.8,393.2775,193,5.968488,567
37,393.6,3.2,385.8,7.3,390.3,4.4,12688.2,783.6,395.5875,240,0.003039,888
9,392.5,2.5,379.8,5.0,385.4,4.2,13640.2,869.1,393.2225,65,1.250592,684
21,390.3,5.8,379.4,6.2,384.3,4.2,13184.8,1328.0,398.5850,231,1.240045,648
24,386.7,5.0,381.6,5.2,382.1,6.9,12906.2,677.4,395.3400,250,2.344906,572
6,377.5,15.5,352.8,11.1,369.4,15.2,13608.8,530.5,401.7750,37,0.003192,868


In [6]:
# --- 6. Anotar los 3 mejores (promedio de 5 semillas) en resultados_experimentos.csv ---
ruta_res = "resultados_experimentos.csv"
res = pd.read_csv(ruta_res, sep=";", decimal=",", encoding="utf-8-sig")
res = res[~res["descripcion"].str.startswith("z005")]   # sin duplicar si se vuelve a correr

for trial, r in ranking.head(3).iterrows():
    nueva = {
        "fecha": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
        "descripcion": (f"z005 Optuna trial {trial}: leaves {int(r['params_num_leaves'])}, "
                        f"hessian {r['params_min_sum_hessian_in_leaf']:.4f}, árboles {int(r['user_attrs_best_iter'])} (prom. 5 semillas)"),
        "meses_train": str(MESES_TRAIN),
        "mes_valid": MES_VALID,
        "semilla": str(SEMILLAS),
        "params": str(params_fijos),
        "mejor_n": int(r["mejor_n_mean"]),
        "ganancia_max_M": r["ganancia_max_M_mean"],
    }
    res = pd.concat([res, pd.DataFrame([nueva])], ignore_index=True)

res["n_enviados"] = res["n_enviados"].astype("Int64")
res.to_csv(ruta_res, sep=";", decimal=",", index=False, encoding="utf-8-sig")
res.tail(3)

,fecha,descripcion,meses_train,mes_valid,semilla,params,n_corte_0025,ganancia_0025_M,mejor_n,ganancia_max_M,prob_en_corte,auc,ks,lift,captura,n_enviados,ganancia_publica_M,ganancia_top10000_M,ganancia_top12000_M
13,2026-10-07 20:02,"z005 Optuna trial 34: leaves 137, hessian 9.75...",[202104],202106,"[501419, 623353, 786349, 834569, 946331]","{'objective': 'binary', 'metric': 'auc', 'lear...",NaN,NaN,13212.0,399.6,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN
14,2026-10-07 20:02,"z005 Optuna trial 36: leaves 61, hessian 7.745...",[202104],202106,"[501419, 623353, 786349, 834569, 946331]","{'objective': 'binary', 'metric': 'auc', 'lear...",NaN,NaN,12291.0,395.5,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN
15,2026-10-07 20:02,"z005 Optuna trial 27: leaves 241, hessian 5.63...",[202104],202106,"[501419, 623353, 786349, 834569, 946331]","{'objective': 'binary', 'metric': 'auc', 'lear...",NaN,NaN,12740.0,394.6,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN
